# Analyze Synthetic Data with Heartwood on Terra

This notebook is part of the Heartwood open-source project. Copyright and license metadata are declared in `REUSE.toml`.

This step-by-step tutorial analyzes synthetic data without leaving the current Terra project. The notebook, interactive terminal, and browser interface all use the same Heartwood configuration, conversation, approval decisions, and audit history.

Use the no-weight `ghcr.io/schmiedmayerlab/heartwood:0.2.0-beta.2-terra` image for a hosted model or portable CPU test, or `ghcr.io/schmiedmayerlab/heartwood:0.2.0-beta.2-terra-gpu-nvidia` for NVIDIA local-model inference. Both images contain no model weights and no credentials. Complete the initial setup in the [Terra guide](terra-jupyter-demo.md) before running this notebook. Keep all tutorial data synthetic; this workflow does not authorize a model, dataset, or use of controlled data.


## 1. Configure and Start Heartwood

The directory containing this notebook is the Heartwood project. Open a Terra terminal in this directory and run `heartwood doctor`, then `heartwood`, to complete initial setup. Choose an authorized Stanford AI API Gateway, OpenAI, or Anthropic connection, or select a local model. The image contains no model weights, so a local model must be downloaded explicitly.

Start `heartwood serve` for a hosted or already-running model. Start `heartwood launch --web` for a downloaded local model so Heartwood can load and supervise its inference server. For example:

```bash
heartwood launch --web
```

Keep that terminal running while using this notebook. The command prints the authenticated Terra browser route when Terra exposes the required proxy metadata. Startup can take several minutes for a local model; the terminal reports download, memory, model-loading, and readiness progress. The first code cell below verifies the current project and displays the same full authenticated `/proxy/<Google project>/<cluster>/jupyter/proxy/8767/` browser link when it is available. The pull-request integration uses a deterministic loopback fixture through the real OpenHands SDK and makes no model-quality claim.


## 2. Open the Current Project

The setup cell treats the notebook directory as the project and verifies Terra persistence, configuration, credentials, and model availability before copying two synthetic tables into `input`. It binds the `terra-demo` session identifier used by later cells and displays an authenticated browser link when Terra provides complete proxy metadata. The session appears in the other interfaces after Step 3 creates its first event.


In [ ]:
import json
from html import escape
from pathlib import Path
from shutil import copy2

from IPython.display import HTML
from IPython.display import display as ipython_display

from heartwood.gateway import ModelCatalogError
from heartwood.notebook import (
    NotebookSession,
    build_widget_spec,
    has_authenticated_jupyter_proxy,
    jupyter_proxy_url,
)

project_root = Path.cwd().resolve()
session_id = "terra-demo"
session = NotebookSession(session_id=session_id)
readiness = session.project_readiness()
failed_checks = [check for check in readiness["checks"] if check["status"] == "fail"]
if failed_checks:
    details = "; ".join(str(check["summary"]) for check in failed_checks)
    raise RuntimeError(f"Heartwood project is not ready: {details}")
if readiness["state"] == "setup-required":
    raise RuntimeError("Complete `heartwood` setup in this project before continuing.")
runtime_verified = False
if readiness["state"] == "compute-required":
    try:
        session.discover_models("local", refresh=True)
    except ModelCatalogError as error:
        raise RuntimeError(
            "The selected local model server is not reachable; "
            "keep `heartwood launch --web` running."
        ) from error
    runtime_verified = True
elif readiness["state"] != "ready":
    raise RuntimeError(f"Unexpected Heartwood readiness state: {readiness['state']}")

input_root = project_root / "input"
fixture_root = Path("/opt/heartwood/fixtures/synthetic/omop-like")
input_root.mkdir(parents=True, exist_ok=True)
for filename in ("person.csv", "condition_occurrence.csv"):
    copy2(fixture_root / filename, input_root / filename)

if has_authenticated_jupyter_proxy():
    proxy_url = jupyter_proxy_url(port=8767)
    link = (
        f'<a href="{escape(proxy_url, quote=True)}" target="_blank" '
        'rel="noopener noreferrer">Open Heartwood in a new tab</a>'
    )
    ipython_display(HTML(link))
else:
    print("Terra proxy metadata is unavailable; continue in this notebook or the CLI.")
print("Heartwood project:", project_root)
print("Platform:", readiness["platform_id"])
print("Readiness:", "ready (local runtime verified)" if runtime_verified else "ready")
print("Localized inputs:", sorted(path.name for path in input_root.iterdir()))

## 3. Inspect the Synthetic Project

Ask Heartwood to identify the available dataset before any agent action is proposed.


In [ ]:
detection = session.detect()

print("Dataset proposals")
for proposal in detection.dataset_proposals:
    print(f"- {proposal.dataset_type} ({proposal.confidence:.2f})")
    for item in proposal.evidence:
        print(f"  - {item}")

print("Latest activity:", detection.activity[-1])

## 4. Ask Heartwood to Analyze the Synthetic Data

This task requests an aggregate cohort result through the same OpenHands-backed session used by the terminal and browser. It does not approve or execute a proposed action automatically. The exact result is a capability check for the selected model; reject unexpected actions rather than forcing a model that does not produce the repository-verified plan.


In [ ]:
prompt = (
    "Build the synthetic target-condition cohort for concept 201826 with the "
    "repository-verified cohort Skill. Read the localized tables in input, require "
    "age 18 or older, apply an aggregate count floor of 20, write "
    "cohort-summary.json, and report quality checks without row-level values."
)
run = session.run(prompt)

print("Policy decisions")
for policy in run.policy_status:
    print(f"- {policy.decision}: {policy.endpoint} ({policy.reason})")

print("Approval controls")
for control in run.approval_controls:
    print(f"- {control.target_type}: {control.target_id} [{control.decision or 'pending'}]")

print("Activity")
for item in run.activity:
    print(f"{item.sequence:03d} {item.kind}: {item.detail}")

## 5. Review and Decide

Review every member of the pending OpenHands action set before running the next cell. You may inspect the same pending set in the browser or CLI, but do not decide it there: this notebook's next cell owns the decision. Continue only when the complete set invokes the repository-verified cohort script, reads `input`, writes `cohort-summary.json` inside this project, and does not transmit data. OpenHands returns this as one grouped action set, so **Allow all once** applies to every displayed member. Stop and use **Reject all** in the terminal or browser instead when any member is unexpected.


In [ ]:
pending = [control for control in run.approval_controls if control.decision is None]
if len(pending) != 1:
    raise RuntimeError(f"Expected one pending synthetic action, found {len(pending)}")
completed = session.approve(tool_call_id=pending[0].target_id)

cohort_path = project_root / "cohort-summary.json"
cohort = json.loads(cohort_path.read_text(encoding="utf-8"))
assert cohort["summary"]["source_participant_count"] == 24
assert cohort["summary"]["source_condition_occurrence_count"] == 39
assert cohort["summary"]["participant_count"] == 20
assert cohort["summary"]["condition_occurrence_count"] == 35
assert cohort["quality_checks"]["row_values_exported"] is False
print(json.dumps(cohort, indent=2, sort_keys=True))

## 6. Replay and Export the Audit Record

The next cell reconstructs the persisted session and creates Heartwood's scrubbed JSON Lines audit export. Prompts, responses, paths, action summaries, secrets, and row values are excluded from the default export.


In [ ]:
exported = session.audit_export()
replayed = session.replay()

print("Persisted events:", replayed.event_count)
print("Exports")
for action in exported.export_actions:
    print(f"- {action.label}: {action.path}")

print("Notebook widget sections:", [section.title for section in build_widget_spec(replayed)])

## 7. Continue in the Terminal or Browser

The CLI and browser interface read the same persisted events. Open the `terra-demo` session in the browser, then run this command from a Terra terminal in the same project:

```bash
heartwood --session-id terra-demo replay
```

The notebook, CLI, and browser interface should show the same researcher task, route decision, proposed action group, approval, successful tool outcome, and final model response. Use the interfaces sequentially for one session; independently running writers to the same file-backed session are not supported. See [Use Heartwood on Terra](terra-jupyter-demo.md) for model choices, browser routing, local-model startup, persistence, and troubleshooting.
